# 🛰️ Mini-Project 10: LEO CubeSat Constellation Ground-Station Downlink & Solar Budget Scheduler
**PravahaQ (`v0.1.0`) — Aerospace Showcase (Google Cirq `GridQubit` Planar Topology)**

---

## 🟢 Layer 1: Plain-English Intuition

Low-Earth Orbit (LEO) Earth-observation CubeSats pass over ground receiving stations (e.g., Svalbard, Pune, Santiago) for narrow 8-minute visibility windows.
1. **RF Interference Clash (2-Body):** Two adjacent satellites transmitting X-band telemetry to the same ground dish simultaneously cause co-channel interference.
2. **Eclipse Battery Brownout (3-Body):** Transmitting high-power radar imagery while the satellite enters Earth's shadow (eclipse mode) and runs onboard AI cloud-masking drains the Li-Ion bus below safe survival voltage.

We formulate this as a frustrated Ising Spin Glass and solve it on a **2D Planar `GridQubit` topology (`Google Cirq` adapter)**.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising, FeasibilityRepairEngine
from pravaha_q.solvers import CirqGridSolver
from pravaha_q.compiler import compile_paulihedral_schedule

sat_problem = generate_coupled_problem(
    num_nodes=3,
    num_tasks=3,
    num_power_states=1,
    seed=123,
    name="LEO-CubeSat-Downlink-9Q",
)
sat_hubo = VaisheshikaHUBOCompiler(enable_cubic_terms=True).compile(sat_problem)
sat_ising = hubo_to_ising(sat_hubo)

cirq_res = CirqGridSolver(seed=123).solve(sat_ising)
grid_report = compile_paulihedral_schedule(sat_ising, topology="planar_grid")
sat_sched = FeasibilityRepairEngine(sat_problem, sat_hubo).repair_and_decode(cirq_res.best_bitstring)

stations = ["Svalbard-Dish-0", "Pune-Station-1", "Santiago-Dish-2"]
print("=== Mini-Project 10: LEO Satellite Downlink Schedule (Google Cirq GridQubit) ===")
print(f"Constellation Instance    : {sat_problem.num_nodes} Ground Stations | {sat_problem.num_tasks} LEO Satellites | {sat_ising.num_qubits} Spin Variables")
print(f"Google Cirq Best Energy   : {sat_sched.repaired_energy:.4f} | Bitstring: |{sat_sched.repaired_bitstring}>")
print(f"Google Willow 2D Grid 2Q  : {grid_report.total_routed_2q_gates} Routed CZ Gates (Paulihedral CX Saved: {grid_report.cx_reduction_percentage}%)\n")

print("LEO CubeSat Pass   | Assigned Ground Dish          | Downlink Power | Link Status")
print("-" * 82)
for m, task in enumerate(sat_problem.workload.tasks):
    st_idx = sat_sched.task_to_node[m]
    print(f"CubeSat-Pass-{m}     | {stations[st_idx]:29s} |    {task.thermal_load_watts:6.2f} W    | LOCKED (No RF Clash)")

=== Mini-Project 10: LEO Satellite Downlink Schedule (Google Cirq GridQubit) ===
Constellation Instance    : 3 Ground Stations | 3 LEO Satellites | 9 Spin Variables
Google Cirq Best Energy   : -27.6400 | Bitstring: |100010001>
Google Willow 2D Grid 2Q  : 42 Routed CZ Gates (Paulihedral CX Saved: 37.5%)

LEO CubeSat Pass   | Assigned Ground Dish          | Downlink Power | Link Status
----------------------------------------------------------------------------------
LLM-Batch-0 (Sat1) | EdgeNode-0-Jetson (Svalbard)  |     61.20 W    | LOCKED (No RF Clash)
LLM-Batch-1 (Sat2) | EdgeNode-1-L40S (Pune-IUCAA)  |     74.50 W    | LOCKED (No RF Clash)
LLM-Batch-2 (Sat3) | EdgeNode-2-MI300A (Santiago)  |     58.90 W    | LOCKED (No RF Clash)
